# 07 — GTZAN Experiment (Experiment 2)

Bewertet alle drei Fingerprinting-Systeme auf dem **GTZAN-Datensatz** (1 000 Songs, 10 Genres, je 30 Sekunden).

Im Gegensatz zum Live-Run sind **alle 1 000 Songs gleichzeitig Referenz-DB und Query-Quelle** (`is_ood=False` immer). Dieser Aufbau prüft die **In-Corpus-Robustheit**: Wie gut erkennen die Systeme verzerrte Versionen aus einer kompakten, genrediversen DB?

**Pfade:**
- GTZAN-Audio: `data/gtzan/`
- Queries:     `data/queries/gtzan/`
- Ergebnisse:  `results/gtzan/`
- NeuralFP:    Checkpoint `checkpoints/live_run/` + τ=0.54 aus `data/partitions/neuralFP_threshold.json`

**Abhängigkeiten:** NB 00/01 (`data/partitions/musan_split.json`), NB 04 (`checkpoints/live_run/pfann_model/model.pt`)

## 1. Setup & Imports

In [ ]:
# ── Reproducibility ─────────────────────────────────────────────────────
import random, os, sys, json, time
import numpy as np
random.seed(42)
np.random.seed(42)

from pathlib import Path
PROJECT_ROOT = Path("/home/jupyter/liverun/").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

from query_generation import generate_track_queries, build_manifest
from shazam_pipeline import build_shazam_index, run_shazam_query
from quad_pipeline import build_quad_index, run_quad_query, get_true_scales
from neural_fp import export_query_list
from gtzan_helpers import (
    build_gtzan_metadata_df, export_gtzan_pfann_ref_list,
    build_path_to_id_mapping, match_gtzan_queries_timed,
)
from metrics import (
    classify_result, compute_hit_rate, compute_precision,
    compute_specificity, compute_time_stats,
    compute_scale_estimation_error,
)


In [ ]:
GTZAN_DIR   = PROJECT_ROOT / 'data' / 'gtzan' / 'Data' / 'genres_original'
QUERY_DIR   = PROJECT_ROOT / 'data' / 'queries' / 'gtzan'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'gtzan'
PLOTS_DIR   = RESULTS_DIR / 'plots'
PARTS_DIR   = PROJECT_ROOT / 'data' / 'partitions'
PFANN_DIR   = PROJECT_ROOT / 'src' / 'pfann'
PFANN_LISTS = PFANN_DIR / 'lists'
MANIFEST_PATH = QUERY_DIR / 'manifest_gtzan.csv'

QUERY_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SR = 8000
SCORE_THRESHOLD = 0.54   # aus data/partitions/neuralFP_threshold.json

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})
SYSTEMS       = ['shazam', 'quad', 'neuralFP']
SYSTEM_COLORS = {'shazam': '#1f77b4', 'quad': '#ff7f0e', 'neuralFP': '#2ca02c'}
SYSTEM_LABELS = {'shazam': 'Shazam', 'quad': 'Quad', 'neuralFP': 'NeuralFP'}

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'GTZAN_DIR    = {GTZAN_DIR}')
print(f'RESULTS_DIR  = {RESULTS_DIR}')
print(f'SCORE_THRESHOLD = {SCORE_THRESHOLD}')


## 2. GTZAN laden & prüfen

GTZAN: 10 Genres × 100 Songs = 1 000 WAVs à 30 s, 22 050 Hz mono.
Track-ID-Zuweisung (alphabetische Genre-Reihenfolge): blues 0–99, classical 100–199, …, rock 900–999.

**Wichtig:** Alle 1 000 Songs sind Referenz-DB UND Query-Quelle. Es gibt keine OOD-Songs — `is_ood=False` immer.

In [ ]:
gtzan_df = build_gtzan_metadata_df(GTZAN_DIR)
print(f'GTZAN tracks geladen: {len(gtzan_df)}')
print(f'Genres ({gtzan_df["genre"].nunique()}): {sorted(gtzan_df["genre"].unique())}')
print(gtzan_df.head(3).to_string())


In [ ]:
# Prüfe: alle 1 000 Dateien vorhanden
missing_files = [tid for tid, row in gtzan_df.iterrows()
                 if not Path(row['filepath']).exists()]
if missing_files:
    print(f'FEHLER: {len(missing_files)} Dateien fehlen — z.B. {missing_files[:3]}')
    raise FileNotFoundError(f'{len(missing_files)} GTZAN-Dateien nicht gefunden')
print(f'Alle {len(gtzan_df)} GTZAN-Dateien vorhanden. ✓')

# Prüfe: Länge ≥ 10s und keine Korruption via soundfile
import soundfile as sf
short_tracks = []
corrupt_tracks = []
for tid, row in gtzan_df.iterrows():
    try:
        info = sf.info(str(row['filepath']))
        if info.duration < 10.0:
            short_tracks.append((tid, round(info.duration, 2)))
    except Exception as exc:
        corrupt_tracks.append((tid, str(exc)))
if short_tracks:
    print(f'WARNUNG: {len(short_tracks)} Tracks < 10s: {short_tracks[:5]}')
else:
    print(f'Alle {len(gtzan_df)} Tracks ≥ 10s. ✓')
if corrupt_tracks:
    print(f'WARNUNG: {len(corrupt_tracks)} Dateien korrupt: {corrupt_tracks[:3]}')
else:
    print('Keine Korruptionen festgestellt. ✓')


In [ ]:
# Partition speichern: data/partitions/gtzan_ref.json
ref_ids = gtzan_df.index.tolist()
gtzan_ref_json = PARTS_DIR / 'gtzan_ref.json'
with open(gtzan_ref_json, 'w') as fh:
    json.dump(ref_ids, fh)
print(f'Gespeichert: {gtzan_ref_json} ({len(ref_ids)} IDs)')

# Sanity: 1000 tracks, 10 Genres × 100
assert len(gtzan_df) == 1000, f'Erwartet 1000, erhalten {len(gtzan_df)}'
per_genre = gtzan_df.groupby('genre').size()
assert (per_genre == 100).all(), f'Nicht alle Genres haben 100 Songs:\n{per_genre}'
print('Shape:', gtzan_df.shape)
print(gtzan_df.head(5).to_string())


## 3. Query-Generierung

Wendet dieselbe Distortion-Pipeline wie NB 01 an (Gruppen A–D, 33 Conditions pro Song). Keine OOD-Queries — alle 1 000 Songs sind In-DB.

MUSAN-Eval-Dateien werden für Gruppen B, C, D benötigt (aus `data/partitions/musan_split.json`).

In [ ]:
# MUSAN eval-Dateien laden (für Gruppen B, C, D)
musan_split_path = PARTS_DIR / 'musan_split.json'
assert musan_split_path.exists(), f'musan_split.json fehlt: {musan_split_path}'
with open(musan_split_path) as fh:
    musan_split = json.load(fh)
musan_eval_files = musan_split['eval']
ir_files = musan_eval_files   # Pseudo-IR via MUSAN eval (wie im Live-Run)

print(f'MUSAN eval-Dateien: {len(musan_eval_files)}')
print(f'IR-Dateien:         {len(ir_files)} (Pseudo-IR)')
print(f'Output-Verzeichnis: {QUERY_DIR}')

# Laufzeit-Schätzung (1000 Songs × ~2.5 s/Song × 1 Pass = In-DB only)
import shutil
est_h = len(gtzan_df) * 2.5 / 3600
_, _, free_bytes = shutil.disk_usage(str(QUERY_DIR.parent.parent))
free_gb = free_bytes / 1024**3
est_gb = len(gtzan_df) * 33 * 8000 * 10 * 4 / 1024**3   # 10s WAV float32
print(f'\nLaufzeit-Schätzung: ~{est_h:.1f}h')
print(f'Disk-Schätzung:     ~{est_gb:.1f} GB für {len(gtzan_df)*33} WAVs')
print(f'Freier Speicher:    {free_gb:.1f} GB  '
      f'{"✓" if free_gb > est_gb * 1.5 else "WARNING: eventuell knapp"}')


In [ ]:
# Queries generieren (alle 1000 GTZAN Songs, is_ood=False)
all_rows = []
skipped = []
for tid, row in tqdm(gtzan_df.iterrows(), total=len(gtzan_df), desc='GTZAN queries'):
    rows = generate_track_queries(
        track_id=int(tid),
        is_ood=False,
        filepath=str(row['filepath']),
        out_dir=str(QUERY_DIR),
        musan_eval_files=musan_eval_files,
        ir_files=ir_files,
        sr=SR,
    )
    if rows:
        all_rows.extend(rows)
    else:
        skipped.append(int(tid))
print(f'Queries generiert: {len(all_rows)}  \nÜbersprungen: {len(skipped)}')
if skipped:
    print(f'  Übersprungene Track-IDs: {skipped}')


In [ ]:
# Manifest aufbauen und speichern
manifest = build_manifest(all_rows)

# MANIFEST_PATH already defined in setup cell
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Manifest gespeichert: {MANIFEST_PATH}')

# Sanity checks
n_songs_ok = len(gtzan_df) - len(skipped)
expected_total = n_songs_ok * 33   # A:15 + B:8 + C:2 + D:8
assert len(manifest) == expected_total, (
    f'Erwartet {expected_total}, erhalten {len(manifest)}'
)
assert manifest['is_ood'].sum() == 0, 'GTZAN hat keine OOD-Songs!'
assert (manifest['ref_track_id'] == manifest['track_id']).all(), (
    'ref_track_id != track_id für In-DB-Queries!'
)

# Speed-Bedingungen müssen von 10.0s abweichen
speed_conds = manifest[manifest['condition'].str.startswith('A_speed_')]
assert not (speed_conds['duration_sec'] == 10.0).any(), (
    'Speed-Conditions dürfen nicht normiert sein!'
)
print(f'Anzahl-Check: {len(manifest)} = {n_songs_ok} Songs × 33 ✓')
print(f'is_ood=False überall: {manifest["is_ood"].sum()} OOD-Einträge ✓')
print(f'Speed-Längen weichen ab ✓')
print(f'\nShape: {manifest.shape}')
print(manifest.head(5).to_string())


## 4. Shazam auf GTZAN

Index auf allen 1 000 GTZAN-Songs aufbauen. Muster identisch zu NB 02 — Imports aus `src/shazam_pipeline.py`.

**Voraussetzung:** Zelle 3 muss vollständig ausgeführt worden sein (manifest_gtzan.csv vorhanden).

In [ ]:
# Voraussetzungen prüfen
assert MANIFEST_PATH.exists(), f'Manifest fehlt: {MANIFEST_PATH}'
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest geladen: {len(manifest)} Queries')

# Laufzeit-Schätzung Shazam
n_ref = len(gtzan_df)
n_queries = len(manifest)
print(f'Index-Build:  ~{n_ref * 1.0 / 60:.0f} min ({n_ref} Songs × ~1 s/Song)')
print(f'Query-Run:    ~{n_queries * 0.35 / 60:.0f} min ({n_queries} Queries × ~350 ms/Query)')


In [ ]:
# Shazam-Index aufbauen
print(f'Baue Shazam-Index auf {n_ref} GTZAN-Songs ...')
t_build_start = time.perf_counter()
shazam_db, build_stats = build_shazam_index(ref_ids, gtzan_df)
build_time_s = time.perf_counter() - t_build_start

db_stats = shazam_db.get_stats()
index_size_mb = db_stats['memory_mb']
print(f'Build-Zeit:   {build_time_s:.1f} s')
print(f'Index-Größe:  {index_size_mb:.1f} MB')
print(f'Hashes total: {build_stats["total_hashes"]:,}')
print(f'Fehlgeschlagen: {build_stats["failed"]}')


In [ ]:
# Alle Queries ausführen
raw_rows_shazam = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='Shazam GTZAN'):
    pred_id, score, q_ms = run_shazam_query(row.query_path, shazam_db)
    ref_id = None if pd.isna(row.ref_track_id) else int(row.ref_track_id)
    result_class = classify_result(pred_id, ref_id, bool(row.is_ood))
    raw_rows_shazam.append({
        'system':        'shazam',
        'track_id':      int(row.track_id),
        'ref_track_id':  ref_id,
        'is_ood':        bool(row.is_ood),
        'predicted_id':  pred_id,
        'score':         score,
        'result_class':  result_class,
        'query_time_ms': q_ms,
        'group':         row.group,
        'condition':     row.condition,
        'duration_sec':  float(row.duration_sec),
    })
print(f'Queries abgeschlossen: {len(raw_rows_shazam)}')


In [ ]:
# Ergebnisse speichern
shazam_df = pd.DataFrame(raw_rows_shazam)
shazam_df['predicted_id'] = shazam_df['predicted_id'].astype(pd.Int64Dtype())
shazam_df['ref_track_id'] = shazam_df['ref_track_id'].astype(pd.Int64Dtype())

shazam_csv = RESULTS_DIR / 'shazam_gtzan_raw.csv'
shazam_df.to_csv(shazam_csv, index=False)
print(f'Gespeichert: {shazam_csv} ({len(shazam_df)} Zeilen)')

# Effizienz-JSON
shazam_eff = {
    'system':          'shazam',
    'dataset':         'gtzan',
    'build_time_s':    round(build_time_s, 3),
    'index_size_mb':   round(index_size_mb, 3),
    'n_ref_tracks':    build_stats['processed'],
    'total_hashes':    build_stats['total_hashes'],
    'mean_query_ms':   round(shazam_df['query_time_ms'].mean(), 3),
    'median_query_ms': round(shazam_df['query_time_ms'].median(), 3),
}
with open(RESULTS_DIR / 'shazam_gtzan_efficiency.json', 'w') as fh:
    json.dump(shazam_eff, fh, indent=2)
print('Effizienz gespeichert.')

# Sanity checks
classes = shazam_df['result_class'].value_counts()
print(f'Ergebnisklassen: {dict(classes)}')
for cls in ['TP', 'FN', 'FP']:
    if cls in classes: print(f'  {cls}: {classes[cls]} ✓')
    else: print(f'  WARNUNG: Klasse {cls} fehlt!')
# GTZAN hat keine OOD-Songs — TN wird nicht erwartet
assert shazam_df['is_ood'].sum() == 0, 'Unerwartete OOD-Einträge!'
# pred_id >= 0 (track_id=0 ist gültig in GTZAN)
invalid = shazam_df['predicted_id'].dropna()
assert (invalid >= 0).all(), f'Ungültige predicted_id < 0: {invalid[invalid < 0].tolist()}'
print(f'\nShape: {shazam_df.shape}')
print(shazam_df.head(5).to_string())


## 5. Quad auf GTZAN

Analog zu Zelle 4, Muster wie NB 03 — Imports aus `src/quad_pipeline.py`.
Quad-spezifisch: `detected_time_scale`, `detected_freq_scale` werden mitgeloggt.

In [ ]:
# Laufzeit-Schätzung Quad
from quad_fingerprint import config
print(f"SEARCH_RADIUS: {config.SEARCH_RADIUS}")
print(f"VERIFICATION_THRESHOLD: {config.VERIFICATION_THRESHOLD}")


# Quad-Index aufbauen
t_build_start_q = time.perf_counter()
quad_db, quad_build_stats = build_quad_index(ref_ids, gtzan_df)
quad_build_time_s = time.perf_counter() - t_build_start_q
print(json.dumps(quad_build_stats, indent=2))


In [ ]:
# Alle Queries ausführen
raw_rows_quad = []
t_wall_start_q = time.perf_counter()

for _, qrow in tqdm(manifest.iterrows(), total=len(manifest), desc='Quad GTZAN'):
    track_id     = int(qrow['track_id'])
    query_path   = qrow['query_path']
    ref_track_id = None if pd.isna(qrow['ref_track_id']) else int(qrow['ref_track_id'])
    is_ood       = bool(qrow['is_ood'])
    condition    = qrow['condition']

    pred_id, score, q_ms, det_t, det_f = run_quad_query(query_path, quad_db)
    result_class = classify_result(pred_id, ref_track_id, is_ood)
    true_t, true_f = get_true_scales(condition)

    raw_rows_quad.append({
        'system':               'quad',
        'track_id':             track_id,
        'ref_track_id':         ref_track_id,
        'is_ood':               is_ood,
        'predicted_id':         pred_id,
        'score':                score,
        'result_class':         result_class,
        'query_time_ms':        q_ms,
        'group':                qrow['group'],
        'condition':            condition,
        'duration_sec':         qrow['duration_sec'],
        'detected_time_scale':  det_t,
        'detected_freq_scale':  det_f,
        'true_time_scale':      true_t,
        'true_freq_scale':      true_f,
    })

total_wall_ms_q = (time.perf_counter() - t_wall_start_q) * 1000.0
print(f'Queries: {len(raw_rows_quad)} total, {total_wall_ms_q/1000:.1f}s wall time')


In [ ]:
# Ergebnisse speichern
quad_df = pd.DataFrame(raw_rows_quad)
quad_df['ref_track_id'] = quad_df['ref_track_id'].astype(pd.Int64Dtype())
quad_df['predicted_id'] = quad_df['predicted_id'].astype(pd.Int64Dtype())

quad_csv = RESULTS_DIR / 'quad_gtzan_raw.csv'
quad_df.to_csv(quad_csv, index=False)
print(f'Gespeichert: {quad_csv} ({len(quad_df)} Zeilen)')

# Effizienz-JSON
quad_eff = {
    'system':        'quad',
    'dataset':       'gtzan',
    'build_stats':   quad_build_stats,
    'total_queries': len(raw_rows_quad),
    'total_wall_ms': round(total_wall_ms_q, 3),
    'avg_query_ms':  round(total_wall_ms_q / len(raw_rows_quad), 3) if raw_rows_quad else 0.0,
}
with open(RESULTS_DIR / 'quad_gtzan_efficiency.json', 'w') as fh:
    json.dump(quad_eff, fh, indent=2)
print('Effizienz gespeichert.')

# Sanity checks
print(f'Ergebnisklassen: {dict(quad_df["result_class"].value_counts())}')
invalid_q = quad_df['predicted_id'].dropna()
assert (invalid_q >= 0).all(), f'Ungültige pred_id < 0: {invalid_q[invalid_q < 0].tolist()}'
print(f'\nShape: {quad_df.shape}')
print(quad_df.head(5).to_string())


## 6. NeuralFP auf GTZAN

Derselbe Checkpoint wie im Live-Run (`checkpoints/live_run/pfann_model/model.pt`).
Threshold: τ = 0.54 (aus `data/partitions/neuralFP_threshold.json`).

Da GTZAN-Dateinamen (`blues.00000.wav`) nicht als Integer parsebar sind,
wird `match_gtzan_queries_timed()` aus `src/gtzan_helpers.py` verwendet,
die eine Path→Track-ID-Tabelle nutzt.

In [ ]:
import subprocess

# Voraussetzungen prüfen
CHECKPOINT_DIR   = PROJECT_ROOT / 'checkpoints' / 'live_run' / 'pfann_model'
PFANN_CONFIG     = PFANN_DIR / 'configs' / 'live_run.json'
GTZAN_REF_LIST   = PFANN_LISTS / 'gtzan_ref.txt'
GTZAN_QUERY_LIST = PFANN_LISTS / 'gtzan_queries.txt'
PFANN_DB_DIR     = RESULTS_DIR / 'pfann_db'
NEURAL_CSV       = RESULTS_DIR / 'neuralFP_gtzan_raw.csv'

# Threshold aus JSON laden
threshold_json = PARTS_DIR / 'neuralFP_threshold.json'
assert threshold_json.exists(), f'Threshold-JSON fehlt: {threshold_json}'
with open(threshold_json) as fh:
    thr_record = json.load(fh)
score_threshold_nfp = thr_record.get('score_threshold', SCORE_THRESHOLD)
print(f'Geladener Score-Threshold: {score_threshold_nfp:.4f}')

required = {
    'model.pt':      CHECKPOINT_DIR / 'model.pt',
    'live_run.json': PFANN_CONFIG,
    'manifest':      MANIFEST_PATH,
}
all_ok = True
for label, p in required.items():
    status = 'OK' if Path(p).exists() else 'FEHLT'
    print(f'  [{status}] {label}: {p}')
    if not Path(p).exists():
        all_ok = False
if not all_ok:
    raise FileNotFoundError('Voraussetzungen nicht erfüllt — NB 04 ausführen!')
print('Alle Voraussetzungen erfüllt. ✓')


In [ ]:
# GTZAN pfann-Referenzliste schreiben
export_gtzan_pfann_ref_list(gtzan_df, GTZAN_REF_LIST)

# pfann-DB aufbauen
PFANN_DB_DIR.mkdir(parents=True, exist_ok=True)
python_bin = sys.executable
build_cmd = [python_bin, 'builder.py', str(GTZAN_REF_LIST), str(PFANN_DB_DIR), str(PFANN_CONFIG)]
print(f'Kommando: {" ".join(build_cmd)}')

t0_nfp_build = time.time()
build_result = subprocess.run(
    build_cmd, cwd=str(PFANN_DIR),
    capture_output=True, text=True, check=True,
)
nfp_build_time_s = time.time() - t0_nfp_build
print(f'DB aufgebaut in {nfp_build_time_s:.1f}s')
for line in build_result.stdout.strip().splitlines()[-10:]:
    print(line)

db_files = list(PFANN_DB_DIR.iterdir())
print(f'\nDB-Verzeichnis ({len(db_files)} Dateien):')
for f in sorted(db_files):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')


In [ ]:
# Query-Liste aus Manifest erzeugen (Manifest-Reihenfolge!)
manifest_nfp = pd.read_csv(MANIFEST_PATH)
export_query_list(manifest_nfp, GTZAN_QUERY_LIST)

with open(GTZAN_QUERY_LIST) as fh:
    sample = fh.readlines()
print(f'Erste 3 Query-Pfade:')
for line in sample[:3]:
    print(f'  {line.rstrip()}')
print(f'  ... ({len(sample)} Einträge gesamt)')


In [ ]:
# Path→Track-ID-Mapping aufbauen (GTZAN-spezifisch)
path_to_id = build_path_to_id_mapping(gtzan_df)
print(f'Path-to-ID-Mapping: {len(path_to_id)} Einträge')

# Laufzeit-Schätzung NeuralFP
print(f'Query-Run: ~{n_queries * 0.12 / 60:.0f} min ({n_queries} Queries × ~120 ms/Query)')

# Queries matchen (per-Query FAISS-Timing)
nfp_df = match_gtzan_queries_timed(
    query_list_path = GTZAN_QUERY_LIST,
    db_dir          = PFANN_DB_DIR,
    manifest        = manifest_nfp,
    path_to_id      = path_to_id,
    out_path        = NEURAL_CSV,
    score_threshold = score_threshold_nfp,
    pfann_dir       = PFANN_DIR,
)
print(f'\nShape: {nfp_df.shape}')
print(nfp_df.head(5).to_string())


In [ ]:
# Effizienz-JSON
nfp_times = nfp_df['query_time_ms'].dropna()
nfp_eff = {
    'system':          'neuralFP',
    'dataset':         'gtzan',
    'build_time_s':    round(nfp_build_time_s, 3),
    'score_threshold': score_threshold_nfp,
    'n_ref_tracks':    len(gtzan_df),
    'n_queries':       len(nfp_df),
    'mean_query_ms':   round(float(nfp_times.mean()), 3) if len(nfp_times) > 0 else None,
    'median_query_ms': round(float(nfp_times.median()), 3) if len(nfp_times) > 0 else None,
    'p95_query_ms':    round(float(nfp_times.quantile(0.95)), 3) if len(nfp_times) > 0 else None,
}
with open(RESULTS_DIR / 'neuralFP_gtzan_efficiency.json', 'w') as fh:
    json.dump(nfp_eff, fh, indent=2)
print(f'Effizienz gespeichert.')
print(json.dumps(nfp_eff, indent=2))

# Sanity checks
print(f'\nErgebnisklassen: {dict(nfp_df["result_class"].value_counts())}')
assert nfp_df['is_ood'].sum() == 0, 'Unerwartete OOD-Einträge in NeuralFP!'
invalid_n = nfp_df['predicted_id'].dropna()
assert (invalid_n >= 0).all(), f'Ungültige predicted_id < 0'
print('Sanity checks bestanden. ✓')


## 7. Evaluation

Lädt alle drei Raw-Result-CSVs, kombiniert sie und erstellt dieselben Plots wie NB 06. Da GTZAN keine OOD-Songs enthält, entfällt die Specificity-Auswertung (`is_ood=False` immer).

In [ ]:
# Alle drei Raw-CSVs laden
required_csvs = {
    'shazam_gtzan_raw.csv':   RESULTS_DIR / 'shazam_gtzan_raw.csv',
    'quad_gtzan_raw.csv':     RESULTS_DIR / 'quad_gtzan_raw.csv',
    'neuralFP_gtzan_raw.csv': RESULTS_DIR / 'neuralFP_gtzan_raw.csv',
}
for label, p in required_csvs.items():
    status = 'OK' if p.exists() else 'FEHLT — Zellen 4/5/6 ausführen!'
    print(f'  [{status}] {label}')

df_shazam = pd.read_csv(RESULTS_DIR / 'shazam_gtzan_raw.csv')
df_quad   = pd.read_csv(RESULTS_DIR / 'quad_gtzan_raw.csv')
df_nfp    = pd.read_csv(RESULTS_DIR / 'neuralFP_gtzan_raw.csv')
df_all    = pd.concat([df_shazam, df_quad, df_nfp], ignore_index=True)

print(f'\ndf_all Shape: {df_all.shape}')
print(f'Systeme: {sorted(df_all["system"].unique())}')
print(f'Gruppen: {sorted(df_all["group"].unique())}')
print(f'Conditions: {df_all["condition"].nunique()}')


In [ ]:
# Gruppe A — Hit Rate vs. Scaling
def _condition_info(cond):
    if cond == 'A_original': return ('original', None)
    parts = cond.split('_')
    if len(parts) < 3: return ('unknown', 0)
    dtype = parts[1]
    vstr  = parts[2]
    if dtype == 'pitch':
        sign = -1 if vstr.startswith('m') else 1
        return (dtype, sign * int(vstr[1:]))
    return (dtype, int(vstr))

group_A = df_all[df_all['group'] == 'A'].copy()
group_A['dist_type'] = group_A['condition'].apply(lambda c: _condition_info(c)[0])
group_A['dist_val']  = group_A['condition'].apply(lambda c: _condition_info(c)[1])
orig_rows = group_A[group_A['dist_type'] == 'original'].copy()
sub_copies = []
for dt, xval in [('tempo', 100), ('speed', 100), ('pitch', 0)]:
    cp = orig_rows.copy(); cp['dist_type'] = dt; cp['dist_val'] = xval
    sub_copies.append(cp)
group_A_ext = pd.concat(
    [group_A[group_A['dist_type'] != 'original']] + sub_copies, ignore_index=True
)

dist_meta = {
    'tempo': ('Tempo-Faktor (%)', 'Tempo'),
    'pitch': ('Pitch-Shift (Halbtöne)', 'Pitch'),
    'speed': ('Speed-Faktor (%)', 'Speed'),
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('GTZAN — Gruppe A: Skalierungs-Robustheit', fontsize=12)

for ax, (dtype, (xlabel, title)) in zip(axes, dist_meta.items()):
    sub = group_A_ext[group_A_ext['dist_type'] == dtype]
    for sys_name in SYSTEMS:
        sys_sub = sub[sub['system'] == sys_name]
        if sys_sub.empty: continue
        hr_rows = []
        for val, grp in sys_sub.groupby('dist_val'):
            tp = (grp['result_class'] == 'TP').sum()
            hr_rows.append({'x': val, 'hit_rate': tp / len(grp) if len(grp) > 0 else 0.0})
        hr_df = pd.DataFrame(hr_rows).sort_values('x')
        ax.plot(hr_df['x'], hr_df['hit_rate'], label=SYSTEM_LABELS[sys_name],
                color=SYSTEM_COLORS[sys_name], marker='o', linewidth=1.5)
    ax.set_title(title); ax.set_xlabel(xlabel)
    ax.set_ylabel('Hit Rate'); ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, ls='--', lw=0.8, color='gray')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gtzan_groupA_scaling.png', bbox_inches='tight')
plt.show()
print('Plot gespeichert: gtzan_groupA_scaling.png')


In [ ]:
# Gruppe B — SNR + FX
group_B = df_all[(df_all['group'] == 'B') & (~df_all['is_ood'])].copy()

def _snr_value(cond):
    parts = cond.split('_')
    vstr = parts[-1]
    if vstr.startswith('m'): return -int(vstr[1:])
    return int(vstr) if vstr.isdigit() else None

# SNR-Plot
snr_rows = group_B[group_B['condition'].str.startswith('B_snr')].copy()
snr_rows['snr_db'] = snr_rows['condition'].apply(_snr_value)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('GTZAN — Gruppe B: Rauschen & Effekte', fontsize=12)

ax = axes[0]
ax.set_title('Hit Rate vs. SNR')
for sys_name in SYSTEMS:
    sys_sub = snr_rows[snr_rows['system'] == sys_name]
    if sys_sub.empty: continue
    hr_rows = []
    for snr_val, grp in sys_sub.groupby('snr_db'):
        tp = (grp['result_class'] == 'TP').sum()
        hr_rows.append({'snr_db': snr_val, 'hit_rate': tp / len(grp)})
    hr_df = pd.DataFrame(hr_rows).sort_values('snr_db')
    ax.plot(hr_df['snr_db'], hr_df['hit_rate'], label=SYSTEM_LABELS[sys_name],
            color=SYSTEM_COLORS[sys_name], marker='o', linewidth=1.5)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Hit Rate')
ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=9)

# FX-Plot
fx_conds = ['B_room_ir', 'B_mp3_128', 'B_mp3_64']
fx_rows = group_B[group_B['condition'].isin(fx_conds)]
ax = axes[1]; ax.set_title('Hit Rate: Room-IR & MP3')
x = np.arange(len(fx_conds))
w = 0.25
for i, sys_name in enumerate(SYSTEMS):
    sys_sub = fx_rows[fx_rows['system'] == sys_name]
    hrs = []
    for cond in fx_conds:
        grp = sys_sub[sys_sub['condition'] == cond]
        tp = (grp['result_class'] == 'TP').sum()
        hrs.append(tp / len(grp) if len(grp) > 0 else 0.0)
    ax.bar(x + i * w, hrs, w, label=SYSTEM_LABELS[sys_name], color=SYSTEM_COLORS[sys_name])
ax.set_xticks(x + w); ax.set_xticklabels(fx_conds, rotation=15)
ax.set_ylabel('Hit Rate'); ax.set_ylim(0, 1.05); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gtzan_groupB.png', bbox_inches='tight')
plt.show()
print('Plot gespeichert: gtzan_groupB.png')


In [ ]:
# Gruppe C — Club / Radio
group_C = df_all[(df_all['group'] == 'C') & (~df_all['is_ood'])].copy()

fig, ax = plt.subplots(figsize=(6, 4))
fig.suptitle('GTZAN — Gruppe C: Real-World-Szenarien', fontsize=12)

c_conds = sorted(group_C['condition'].unique())
x = np.arange(len(c_conds))
w = 0.25
for i, sys_name in enumerate(SYSTEMS):
    sys_sub = group_C[group_C['system'] == sys_name]
    hrs = []
    for cond in c_conds:
        grp = sys_sub[sys_sub['condition'] == cond]
        tp = (grp['result_class'] == 'TP').sum()
        hrs.append(tp / len(grp) if len(grp) > 0 else 0.0)
    ax.bar(x + i * w, hrs, w, label=SYSTEM_LABELS[sys_name], color=SYSTEM_COLORS[sys_name])
ax.set_xticks(x + w); ax.set_xticklabels(c_conds)
ax.set_ylabel('Hit Rate'); ax.set_ylim(0, 1.05); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gtzan_groupC.png', bbox_inches='tight')
plt.show()
print('Plot gespeichert: gtzan_groupC.png')


In [ ]:
# Gruppe D — Query-Länge
group_D = df_all[(df_all['group'] == 'D') & (~df_all['is_ood'])].copy()

def _parse_D_condition(cond):
    parts = cond.split('_')
    if len(parts) < 3: return None, None
    dur_str = parts[-1].replace('s', '')
    noise   = '_'.join(parts[1:-1])
    try: return int(dur_str), noise
    except ValueError: return None, None

group_D['nominal_s'] = group_D['condition'].apply(lambda c: _parse_D_condition(c)[0])
group_D['noise_lvl'] = group_D['condition'].apply(lambda c: _parse_D_condition(c)[1])
group_D = group_D.dropna(subset=['nominal_s', 'noise_lvl'])

fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle('GTZAN — Gruppe D: Query-Länge', fontsize=12)
linestyles = {'clean': '-', 'snr10': '--'}

for sys_name in SYSTEMS:
    sys_sub = group_D[group_D['system'] == sys_name]
    for noise, ls in linestyles.items():
        n_sub = sys_sub[sys_sub['noise_lvl'] == noise]
        if n_sub.empty: continue
        hr_rows = []
        for dur, grp in n_sub.groupby('nominal_s'):
            tp = (grp['result_class'] == 'TP').sum()
            hr_rows.append({'dur': dur, 'hit_rate': tp / len(grp)})
        hr_df = pd.DataFrame(hr_rows).sort_values('dur')
        ax.plot(hr_df['dur'], hr_df['hit_rate'],
                label=f'{SYSTEM_LABELS[sys_name]} ({noise})',
                color=SYSTEM_COLORS[sys_name], ls=ls, marker='o', linewidth=1.5)
ax.set_xlabel('Query-Länge (s)'); ax.set_ylabel('Hit Rate')
ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gtzan_groupD.png', bbox_inches='tight')
plt.show()
print('Plot gespeichert: gtzan_groupD.png')


In [ ]:
# Zusammenfassung: Hit Rate pro System × Gruppe
rows_summary = []
for sys_name in SYSTEMS:
    sys_df = df_all[df_all['system'] == sys_name]
    row = {'System': SYSTEM_LABELS[sys_name]}
    for grp in sorted(df_all['group'].unique()):
        hr = compute_hit_rate(sys_df[sys_df['group'] == grp])
        row[f'Gruppe {grp}'] = hr
    row['Gesamt'] = compute_hit_rate(sys_df)
    row['Precision'] = compute_precision(sys_df)
    # Kein Specificity (keine OOD-Songs in GTZAN)
    rows_summary.append(row)

summary_df = pd.DataFrame(rows_summary).set_index('System')
print('=== GTZAN Zusammenfassung ===')
print(summary_df.to_string(float_format=lambda x: f'{x:.3f}'))

summary_df.to_csv(RESULTS_DIR / 'gtzan_summary.csv')
print(f'\nGespeichert: {RESULTS_DIR / "gtzan_summary.csv"}')


In [ ]:
# Effizienz-Tabelle
eff_rows = []
for sys_name, eff_file, sys_df in [
    ('Shazam',    RESULTS_DIR / 'shazam_gtzan_efficiency.json',   df_shazam),
    ('Quad',      RESULTS_DIR / 'quad_gtzan_efficiency.json',     df_quad),
    ('NeuralFP',  RESULTS_DIR / 'neuralFP_gtzan_efficiency.json', df_nfp),
]:
    if not eff_file.exists():
        print(f'  WARNUNG: {eff_file.name} fehlt')
        continue
    with open(eff_file) as fh:
        eff = json.load(fh)
    t = compute_time_stats(sys_df)
    if sys_name == 'Shazam':
        n_ref_e = eff.get('n_ref_tracks', float('nan'))
        build_t = eff.get('build_time_s', float('nan'))
        idx_mb  = eff.get('index_size_mb', float('nan'))
        eff_rows.append({
            'System': sys_name, 'Build-Zeit (s)': build_t,
            'Build/Song (s)': build_t / n_ref_e if n_ref_e else float('nan'),
            'Index-MB': idx_mb,
            'Median-Query (ms)': t['median'], 'P95-Query (ms)': t['p95'],
        })
    elif sys_name == 'Quad':
        bs = eff.get('build_stats', {})
        n_ref_e = bs.get('processed', float('nan'))
        build_t = bs.get('build_time_s', float('nan'))
        idx_mb  = bs.get('db_memory_mb', float('nan'))
        eff_rows.append({
            'System': sys_name, 'Build-Zeit (s)': build_t,
            'Build/Song (s)': build_t / n_ref_e if n_ref_e else float('nan'),
            'Index-MB': idx_mb,
            'Median-Query (ms)': t['median'], 'P95-Query (ms)': t['p95'],
        })
    else:  # NeuralFP
        n_ref_e = eff.get('n_ref_tracks', float('nan'))
        build_t = eff.get('build_time_s', float('nan'))
        eff_rows.append({
            'System': sys_name, 'Build-Zeit (s)': build_t,
            'Build/Song (s)': build_t / n_ref_e if n_ref_e else float('nan'),
            'Index-MB': float('nan'),
            'Median-Query (ms)': t['median'], 'P95-Query (ms)': t['p95'],
        })

eff_df = pd.DataFrame(eff_rows).set_index('System')
print('=== Effizienz-Tabelle (GTZAN) ===')
print(eff_df.to_string(float_format=lambda x: f'{x:.2f}'))


In [ ]:
# Abschluss-Sanity
print('=== Abschluss-Sanity ===')
for sys_name, sys_df in [('shazam', df_shazam), ('quad', df_quad), ('neuralFP', df_nfp)]:
    assert sys_df['is_ood'].sum() == 0, f'{sys_name}: Unerwartete OOD-Einträge'
    classes = set(sys_df['result_class'].unique())
    assert classes.issubset({'TP', 'FP', 'FN', 'TN'}), f'{sys_name}: Unbekannte Klassen: {classes}'
    assert 'TP' in classes, f'{sys_name}: Keine TP-Klasse — System gibt immer None zurück!'
    hit = compute_hit_rate(sys_df, filter_col='condition', filter_val='A_original')
    print(f'  {sys_name}: Hit Rate A_original = {hit:.1%}')

print(f'\ndf_all Shape: {df_all.shape}')
print('Erste 5 Zeilen:')
print(df_all.head(5).to_string())
print('\nAlle Checks bestanden. ✓')
